# preprocess_sarima.py
Builds the SARIMA modeling table from the preprocessed housing panel.

In [4]:
"""Build the SARIMA modeling table from the preprocessed housing panel.

Input: outputs/boston_housing_preprocessed.csv
Output: outputs/sarima_data.csv
"""
from pathlib import Path
import pandas as pd
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
INPUT_FILE  = PROJECT_ROOT / "outputs" / "boston_housing_preprocessed.csv"
OUTPUT_FILE = PROJECT_ROOT / "outputs" / "sarima_data.csv"
START_DATE        = "2015-01-31"
END_DATE          = "2024-12-31"
MAX_ZHVI_NULL_PCT = 0.05

In [5]:
def main():
    print("Preparing SARIMA dataset\n")

    if not INPUT_FILE.exists():
        print(f"[ERROR] {INPUT_FILE} not found. Run preprocess.py first.")
        return

    df = pd.read_csv(INPUT_FILE, dtype={"zip_code": str})
    df["date"] = pd.to_datetime(df["date"])

    df = df[(df["date"] >= START_DATE) & (df["date"] <= END_DATE)].copy()
    print(f"After date filter ({START_DATE} to {END_DATE}): {len(df):,} rows")

    cols = ["zip_code", "date", "zhvi", "location_tier", "MORTGAGE30US"]
    df = df[cols]

    # Drop ZIPs with too many missing ZHVI values
    zhvi_null_pct = df.groupby("zip_code")["zhvi"].apply(
        lambda x: x.isnull().mean()
    )
    valid_zips = zhvi_null_pct[zhvi_null_pct <= MAX_ZHVI_NULL_PCT].index
    dropped = df["zip_code"].nunique() - len(valid_zips)
    df = df[df["zip_code"].isin(valid_zips)].copy()
    print(f"Dropped {dropped} ZIPs with >{MAX_ZHVI_NULL_PCT*100:.0f}% missing ZHVI")

    # Fill any remaining small ZHVI gaps with linear interpolation
    df = df.sort_values(["zip_code", "date"])
    df["zhvi"] = df.groupby("zip_code")["zhvi"].transform(
        lambda x: x.interpolate(method="linear", limit=3)
    )

    df = df.sort_values(["zip_code", "date"]).reset_index(drop=True)

    OUTPUT_FILE.parent.mkdir(exist_ok=True)
    df.to_csv(OUTPUT_FILE, index=False)

    print(f"\n Saved {OUTPUT_FILE}")
    print(f"{len(df):,} rows | {df['zip_code'].nunique()} ZIP codes")
    print(f"Date range: {df['date'].min().date()} -> {df['date'].max().date()}")
    print(f"ZHVI nulls remaining: {df['zhvi'].isnull().sum()}")
    print(f"MORTGAGE30US nulls:   {df['MORTGAGE30US'].isnull().sum()}")

    print("\nLocation tier breakdown:")
    tier_counts = df.drop_duplicates("zip_code")["location_tier"].value_counts()
    for tier, count in tier_counts.items():
        print(f"  {tier}: {count} ZIPs")

In [6]:
if __name__ == "__main__":
    main()

Preparing SARIMA dataset

After date filter (2015-01-31 to 2024-12-31): 32,640 rows
Dropped 5 ZIPs with >5% missing ZHVI

 Saved C:\Users\tanma\Desktop\DS4420\housingAffordability\outputs\sarima_data.csv
32,040 rows | 267 ZIP codes
Date range: 2015-01-31 -> 2024-12-31
ZHVI nulls remaining: 0
MORTGAGE30US nulls:   0

Location tier breakdown:
  outer_suburb: 214 ZIPs
  boston_city: 39 ZIPs
  inner_suburb: 14 ZIPs
